# SHAP — Particle-phase ClPAHs (self-contained)

This notebook explains the particle-phase chlorinated PAHs (ClPAHs) XGBoost model with SHAP.

It is fully self-contained: all data loading, cleaning, training and SHAP functions are defined inline below. Just run the cells top-to-bottom; the last cell reproduces training, then writes SHAP figures and CSVs to `particle_output/shap_particle/`.

Paths are relative: training data = `../Data/Data_S2_Particle_training.csv`; all outputs go under `particle_output/`.


In [ ]:
# -*- coding: utf-8 -*-
"""
颗粒相ClPAHs建模（固定最终参数 + 原始特征SHAP完整输出）
------------------------------------------------------------
用途：
1. 使用固定最终参数直接建模
2. 输出随机RepeatedCV与随机Holdout结果
3. 输出两套SHAP解释：
   - 模型特征层面的SHAP
   - 聚合回原始特征层面的SHAP
4. 输出各种图：
   - CV箱线图
   - Holdout预测散点图
   - Holdout残差图
   - Gain重要性图
   - SHAP summary/bar/top15图
   - 原始特征dependence图
5. 输出误差分层分析
6. 适配当前无城市列的完整CSV

数据结构：
- 第一列：sample_name
- 目标列：concentration
- 其余列：特征
"""

import os
import gc
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import RepeatedKFold, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import xgboost as xgb
import shap


# =========================
# CONFIG
# =========================
FILE_PATH_WINDOWS = r"../Data/Data_S2_Particle_training.csv"
FILE_PATH_FALLBACK = r"../Data/Data_S2_Particle_training.csv"

OUT_DIR = "particle_output"

RANDOM_STATE = 42
TEST_SIZE = 0.20
NTHREAD = 4

MISSING_RATE_THRESHOLD = 0.60
WINSORIZE_Q_LOW = 0.01
WINSORIZE_Q_HIGH = 0.99
CORR_THRESHOLD = 0.985

FINAL_CV_SPLITS = 5
FINAL_CV_REPEATS = 5
NUM_BOOST_ROUND_FINAL_CV = 6500

NUM_BOOST_ROUND_HOLDOUT = 12000
EARLY_STOPPING_ROUNDS = 500
INNER_VALID_SIZE_HOLDOUT = 0.15
USE_HOLDOUT_BAGGING = True
HOLDOUT_BAG_SEEDS = [42, 52, 62, 72, 82, 92, 102, 112, 122]

BEST_PARAMS = {
    "eta": 0.01,
    "max_depth": 5,
    "min_child_weight": 24.0,
    "subsample": 0.9,
    "colsample_bytree": 0.64,
    "gamma": 0.1,
    "alpha": 0.0,
    "lambda_": 10.0,
    "max_delta_step": 0.0
}


# =========================
# Utils
# =========================

In [ ]:
def ensure_outdir(p):
    os.makedirs(p, exist_ok=True)


In [ ]:
def r2_on_log_scale(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.clip(np.asarray(y_pred, float), 0, None)
    return float(r2_score(np.log1p(y_true), np.log1p(y_pred)))


In [ ]:
def metric_pack(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred, float), 0, None)
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "log_R2": r2_on_log_scale(y_true, y_pred),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }


In [ ]:
def load_data():
    fp = FILE_PATH_WINDOWS if os.path.exists(FILE_PATH_WINDOWS) else FILE_PATH_FALLBACK
    if not os.path.exists(fp):
        raise FileNotFoundError(fp)

    for enc in ["utf-8-sig", "gbk", "utf-8"]:
        try:
            return pd.read_csv(fp, encoding=enc), fp
        except Exception:
            pass
    return pd.read_csv(fp), fp


In [ ]:
def to_py(obj):
    if isinstance(obj, dict):
        return {str(k): to_py(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_py(v) for v in obj]
    elif isinstance(obj, tuple):
        return [to_py(v) for v in obj]
    elif hasattr(obj, "item"):
        try:
            return obj.item()
        except Exception:
            return str(obj)
    else:
        return obj


In [ ]:
def safe_num_series(s):
    return pd.to_numeric(
        s.astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce"
    )


In [ ]:
def make_y_bins(y, n_bins=5):
    y = pd.Series(y).reset_index(drop=True)
    n_unique = y.nunique()
    n_bins = min(n_bins, max(2, n_unique))
    try:
        bins = pd.qcut(y, q=n_bins, labels=False, duplicates="drop")
    except Exception:
        bins = pd.qcut(y.rank(method="first"), q=n_bins, labels=False, duplicates="drop")
    return bins.astype(int)


# =========================
# Feature engineering
# =========================

In [ ]:
def add_quarter_features(df):
    if "quarter" not in df.columns:
        return df
    q = pd.to_numeric(df["quarter"], errors="coerce")
    df["quarter_sin"] = np.sin(2 * np.pi * q / 4.0)
    df["quarter_cos"] = np.cos(2 * np.pi * q / 4.0)
    return df


In [ ]:
def add_interaction_features(df):
    cols = set(df.columns)

    if {"PM2.5", "BLH"}.issubset(cols):
        df["PM25_x_BLH"] = df["PM2.5"] * df["BLH"]
    if {"temperature", "humidity"}.issubset(cols):
        df["Temp_x_Humidity"] = df["temperature"] * df["humidity"]
    if {"precipitation", "PM2.5"}.issubset(cols):
        df["Prec_x_PM25"] = df["precipitation"] * df["PM2.5"]
    if {"CO", "SO2"}.issubset(cols):
        df["CO_x_SO2"] = df["CO"] * df["SO2"]
    if {"centralized_heating_or_not", "quarter"}.issubset(cols):
        df["Heating_x_Quarter"] = df["centralized_heating_or_not"] * df["quarter"]
    if {"heavy_industrial_or_not", "PM10"}.issubset(cols):
        df["Industry_x_PM10"] = df["heavy_industrial_or_not"] * df["PM10"]
    if {"PM10", "BLH"}.issubset(cols):
        df["PM10_x_BLH"] = df["PM10"] * df["BLH"]
    if {"NO2", "PM2.5"}.issubset(cols):
        df["NO2_x_PM25"] = df["NO2"] * df["PM2.5"]

    return df


# =========================
# Data cleaning
# =========================

In [ ]:
def clean_and_prepare_data(df):
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all").copy()
    df.columns = [str(c).strip() for c in df.columns]

    cols_lower_map = {str(c).strip().lower(): c for c in df.columns}

    if "sample_name" not in cols_lower_map:
        raise ValueError(f"没有找到 sample_name 列。当前列名为: {df.columns.tolist()}")
    sample_col = cols_lower_map["sample_name"]

    if "concentration" not in cols_lower_map:
        raise ValueError(f"没有找到 concentration 列。当前列名为: {df.columns.tolist()}")
    target_col = cols_lower_map["concentration"]

    raw_feature_cols = [c for c in df.columns if c not in [sample_col, target_col]]
    original_feature_cols = raw_feature_cols.copy()

    work = df[[sample_col, target_col] + raw_feature_cols].copy()
    work[sample_col] = work[sample_col].astype(str).str.strip()
    work[target_col] = safe_num_series(work[target_col])

    for c in raw_feature_cols:
        work[c] = safe_num_series(work[c])

    work = add_quarter_features(work)
    work = add_interaction_features(work)

    raw_n = len(work)

    work = work.dropna(subset=[target_col]).copy()
    after_target_notna = len(work)

    work = work[work[target_col] >= 0].copy()
    after_target_nonneg = len(work)

    work = work[work[sample_col].notna()].copy()
    work = work[work[sample_col].astype(str).str.len() > 0].copy()
    after_sample_clean = len(work)

    feature_cols = [c for c in work.columns if c not in [sample_col, target_col]]
    miss = work[feature_cols].isna().mean()
    feature_cols = miss[miss <= MISSING_RATE_THRESHOLD].index.tolist()
    if not feature_cols:
        raise ValueError("缺失率过滤后无特征。")

    for c in feature_cols:
        s = work[c]
        if int(s.notna().sum()) >= 8:
            lo = s.quantile(WINSORIZE_Q_LOW)
            hi = s.quantile(WINSORIZE_Q_HIGH)
            work[c] = s.clip(lo, hi)

    model_df = work[[sample_col, target_col] + feature_cols].drop_duplicates().copy()
    after_dedup = len(model_df)

    X = model_df[feature_cols].copy()
    y = model_df[target_col].copy()
    meta = model_df[[sample_col]].copy()

    stats = {
        "raw_rows": raw_n,
        "after_target_notna": after_target_notna,
        "after_target_nonnegative": after_target_nonneg,
        "after_sample_clean": after_sample_clean,
        "after_drop_duplicates": after_dedup,
    }

    return X, y, meta, sample_col, target_col, model_df, feature_cols, original_feature_cols, stats


# =========================
# Preprocess
# =========================

In [ ]:
def drop_high_corr_features_train_only(X_train, X_valid=None, thr=0.98):
    if X_train.shape[1] <= 1:
        if X_valid is None:
            return X_train.copy(), None, []
        return X_train.copy(), X_valid.copy(), []

    tmp = X_train.copy().fillna(X_train.median(numeric_only=True))
    corr = tmp.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_cols = [c for c in upper.columns if any(upper[c] > thr)]

    Xtr2 = X_train.drop(columns=drop_cols, errors="ignore").copy()
    if X_valid is None:
        return Xtr2, None, drop_cols

    Xva2 = X_valid.drop(columns=drop_cols, errors="ignore").copy()
    return Xtr2, Xva2, drop_cols


In [ ]:
def fit_preprocess(X_tr):
    imp = SimpleImputer(strategy="median")
    vf = VarianceThreshold(0.0)
    X_imp = imp.fit_transform(X_tr)
    X_f = vf.fit_transform(X_imp)
    kept_mask = vf.get_support()
    kept_cols = [c for c, keep in zip(X_tr.columns.tolist(), kept_mask) if keep]
    return imp, vf, kept_cols


In [ ]:
def transform_preprocess(imp, vf, X):
    return vf.transform(imp.transform(X))


# =========================
# XGB
# =========================

In [ ]:
def train_xgb_log(X_f, y, params, seed, num_boost_round, eval_set=None, early_stopping_rounds=None):
    z = np.log1p(np.asarray(y, float))
    dtr = xgb.DMatrix(X_f, label=z)

    p = dict(
        booster="gbtree",
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        seed=seed,
        nthread=NTHREAD,
        eta=float(params["eta"]),
        max_depth=int(params["max_depth"]),
        min_child_weight=float(params["min_child_weight"]),
        subsample=float(params["subsample"]),
        colsample_bytree=float(params["colsample_bytree"]),
        gamma=float(params["gamma"]),
        reg_alpha=float(params["alpha"]),
        reg_lambda=float(params["lambda_"]),
        max_delta_step=float(params["max_delta_step"]),
    )

    if eval_set is None:
        bst = xgb.train(p, dtr, num_boost_round=num_boost_round, verbose_eval=False)
        return bst, num_boost_round

    dval = xgb.DMatrix(eval_set[0], label=np.log1p(np.asarray(eval_set[1], float)))
    bst = xgb.train(
        p, dtr, num_boost_round=num_boost_round,
        evals=[(dtr, "train"), (dval, "valid")],
        early_stopping_rounds=early_stopping_rounds,
        verbose_eval=False
    )
    best_iter = getattr(bst, "best_iteration", None)
    if best_iter is None:
        best_iter = num_boost_round
    return bst, best_iter


In [ ]:
def predict_xgb_log(bst, X_f):
    z = bst.predict(xgb.DMatrix(X_f))
    return np.clip(np.expm1(z), 0, None)


# =========================
# CV / Holdout
# =========================

In [ ]:
def repeated_cv_eval(X, y, params, n_splits, n_repeats, num_boost_round):
    cv = RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=RANDOM_STATE)

    rows = []
    feature_counter = {}
    dropped_corr_all = []

    for fid, (tr, va) in enumerate(cv.split(X, y), 1):
        X_tr = X.iloc[tr].copy()
        X_va = X.iloc[va].copy()
        y_tr = y.iloc[tr].copy()
        y_va = y.iloc[va].copy()

        X_tr2, X_va2, dropped_corr = drop_high_corr_features_train_only(X_tr, X_va, CORR_THRESHOLD)
        dropped_corr_all.extend(dropped_corr)

        imp, vf, kept_cols = fit_preprocess(X_tr2)
        Xtr_f = transform_preprocess(imp, vf, X_tr2)
        Xva_f = transform_preprocess(imp, vf, X_va2)

        bst, best_iter = train_xgb_log(
            Xtr_f, y_tr, BEST_PARAMS,
            seed=RANDOM_STATE + fid,
            num_boost_round=num_boost_round
        )

        y_pred = predict_xgb_log(bst, Xva_f)
        m = metric_pack(y_va, y_pred)

        rows.append({
            "fold": fid,
            "R2": m["R2"],
            "log_R2": m["log_R2"],
            "MAE": m["MAE"],
            "RMSE": m["RMSE"],
            "best_iteration": best_iter,
            "n_features_after_corr": X_tr2.shape[1],
            "n_features_final": len(kept_cols)
        })

        for c in kept_cols:
            feature_counter[c] = feature_counter.get(c, 0) + 1

        del bst
        gc.collect()

    df_cv = pd.DataFrame(rows)

    feature_freq = pd.DataFrame({
        "feature": list(feature_counter.keys()),
        "selected_times": list(feature_counter.values())
    }).sort_values(["selected_times", "feature"], ascending=[False, True]).reset_index(drop=True)

    if len(dropped_corr_all) > 0:
        dropped_corr_freq = pd.Series(dropped_corr_all).value_counts().reset_index()
        dropped_corr_freq.columns = ["feature", "dropped_times"]
    else:
        dropped_corr_freq = pd.DataFrame(columns=["feature", "dropped_times"])

    return df_cv, feature_freq, dropped_corr_freq


In [ ]:
def holdout_eval_random(X, y, meta, params):
    bins = make_y_bins(y, 5)
    idx = np.arange(len(y))

    tr_idx, te_idx = train_test_split(
        idx, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=bins
    )

    X_train = X.iloc[tr_idx].copy()
    X_test = X.iloc[te_idx].copy()
    y_train = y.iloc[tr_idx].copy()
    y_test = y.iloc[te_idx].copy()
    meta_test = meta.iloc[te_idx].copy()

    X_train2, X_test2, dropped_corr = drop_high_corr_features_train_only(X_train, X_test, CORR_THRESHOLD)

    preds_train = []
    preds_test = []
    best_iters = []
    gain_rows = []
    feature_counter = []

    bag_seeds = HOLDOUT_BAG_SEEDS if USE_HOLDOUT_BAGGING else [RANDOM_STATE + 2026]
    idx2 = np.arange(len(X_train2))

    saved_artifacts = None

    for i, s in enumerate(bag_seeds, 1):
        tr2, va2 = train_test_split(idx2, test_size=INNER_VALID_SIZE_HOLDOUT, random_state=s)

        X_tr = X_train2.iloc[tr2].copy()
        X_va = X_train2.iloc[va2].copy()
        y_tr = y_train.iloc[tr2].copy()
        y_va = y_train.iloc[va2].copy()

        imp, vf, kept_cols = fit_preprocess(X_tr)
        Xtr_f = transform_preprocess(imp, vf, X_tr)
        Xva_f = transform_preprocess(imp, vf, X_va)
        Xtrain_full_f = transform_preprocess(imp, vf, X_train2[kept_cols])
        Xtest_f = transform_preprocess(imp, vf, X_test2[kept_cols])

        bst, best_iter = train_xgb_log(
            Xtr_f, y_tr, params, seed=s,
            num_boost_round=NUM_BOOST_ROUND_HOLDOUT,
            eval_set=(Xva_f, y_va),
            early_stopping_rounds=EARLY_STOPPING_ROUNDS
        )

        pred_train = predict_xgb_log(bst, Xtrain_full_f)
        pred_test = predict_xgb_log(bst, Xtest_f)

        preds_train.append(pred_train)
        preds_test.append(pred_test)
        best_iters.append(best_iter)

        gain_dict = bst.get_score(importance_type="gain")
        weight_dict = bst.get_score(importance_type="weight")
        cover_dict = bst.get_score(importance_type="cover")

        fmap = {f"f{j}": c for j, c in enumerate(kept_cols)}
        used_keys = sorted(set(gain_dict.keys()) | set(weight_dict.keys()) | set(cover_dict.keys()))

        for k in used_keys:
            gain_rows.append({
                "bag_id": i,
                "seed": s,
                "feature": fmap.get(k, k),
                "gain": gain_dict.get(k, 0.0),
                "weight": weight_dict.get(k, 0.0),
                "cover": cover_dict.get(k, 0.0)
            })

        feature_counter.extend(kept_cols)

        saved_artifacts = {
            "bst": bst,
            "imp": imp,
            "vf": vf,
            "kept_cols": kept_cols,
            "X_test2": X_test2,
        }

    y_train_pred = np.mean(np.vstack(preds_train), axis=0)
    y_test_pred = np.mean(np.vstack(preds_test), axis=0)

    selected_features_df = pd.Series(feature_counter).value_counts().reset_index()
    selected_features_df.columns = ["feature", "selected_times_in_bagging"]

    gain_df = pd.DataFrame(gain_rows)
    if len(gain_df) > 0:
        gain_summary = gain_df.groupby("feature", as_index=False).agg({
            "gain": "mean",
            "weight": "mean",
            "cover": "mean"
        }).sort_values("gain", ascending=False).reset_index(drop=True)
    else:
        gain_summary = pd.DataFrame(columns=["feature", "gain", "weight", "cover"])

    return {
        "train_metrics": metric_pack(y_train, y_train_pred),
        "test_metrics": metric_pack(y_test, y_test_pred),
        "y_test": y_test.values,
        "y_test_pred": y_test_pred,
        "meta_test": meta_test,
        "selected_features_df": selected_features_df,
        "gain_summary": gain_summary,
        "best_iterations": best_iters,
        "dropped_corr": dropped_corr,
        "saved_artifacts": saved_artifacts
    }


# =========================
# SHAP aggregation to original features
# =========================

In [ ]:
def build_original_feature_map(original_feature_cols):
    mapper = {}

    for col in original_feature_cols:
        mapper[col] = [(col, 1.0)]

    mapper["quarter_sin"] = [("quarter", 1.0)]
    mapper["quarter_cos"] = [("quarter", 1.0)]

    mapper["PM25_x_BLH"] = [("PM2.5", 0.5), ("BLH", 0.5)]
    mapper["Temp_x_Humidity"] = [("temperature", 0.5), ("humidity", 0.5)]
    mapper["Prec_x_PM25"] = [("precipitation", 0.5), ("PM2.5", 0.5)]
    mapper["CO_x_SO2"] = [("CO", 0.5), ("SO2", 0.5)]
    mapper["Heating_x_Quarter"] = [("centralized_heating_or_not", 0.5), ("quarter", 0.5)]
    mapper["Industry_x_PM10"] = [("heavy_industrial_or_not", 0.5), ("PM10", 0.5)]
    mapper["PM10_x_BLH"] = [("PM10", 0.5), ("BLH", 0.5)]
    mapper["NO2_x_PM25"] = [("NO2", 0.5), ("PM2.5", 0.5)]

    return mapper


In [ ]:
def aggregate_shap_to_original_features(shap_df_model, original_feature_cols, mapper):
    agg = pd.DataFrame(
        0.0,
        index=shap_df_model.index,
        columns=original_feature_cols
    )

    for model_col in shap_df_model.columns:
        if model_col not in mapper:
            continue
        for orig_col, weight in mapper[model_col]:
            if orig_col in agg.columns:
                agg[orig_col] += shap_df_model[model_col] * weight

    return agg


In [ ]:
def plot_top_mean_abs_shap(shap_mean_abs_df, save_path, title, topn=15):
    top_df = shap_mean_abs_df.head(topn).iloc[::-1]
    plt.figure(figsize=(8, max(5, 0.35 * len(top_df))))
    plt.barh(top_df["feature"], top_df["mean_abs_shap"])
    plt.xlabel("Mean |SHAP value|")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()


In [ ]:
def run_shap_analysis_original(saved_artifacts, out_dir, model_df, sample_col, target_col, original_feature_cols):
    bst = saved_artifacts["bst"]
    imp = saved_artifacts["imp"]
    vf = saved_artifacts["vf"]
    kept_cols = saved_artifacts["kept_cols"]
    X_test2 = saved_artifacts["X_test2"]

    # 1) 模型特征层面
    X_model_df = X_test2[kept_cols].copy()
    X_model_f = transform_preprocess(imp, vf, X_model_df)

    explainer = shap.TreeExplainer(bst)
    shap_values_model = explainer.shap_values(X_model_f)

    shap_df_model = pd.DataFrame(shap_values_model, columns=kept_cols)

    model_mean_abs = shap_df_model.abs().mean().sort_values(ascending=False).reset_index()
    model_mean_abs.columns = ["feature", "mean_abs_shap"]
    model_mean_abs.to_csv(
        os.path.join(out_dir, "shap_mean_abs_importance_model_features.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    shap.summary_plot(
        shap_values_model,
        pd.DataFrame(X_model_f, columns=kept_cols),
        show=False
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(out_dir, "shap_summary_model_features.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    shap.summary_plot(
        shap_values_model,
        pd.DataFrame(X_model_f, columns=kept_cols),
        plot_type="bar",
        show=False
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(out_dir, "shap_bar_model_features.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    plot_top_mean_abs_shap(
        model_mean_abs,
        os.path.join(out_dir, "shap_top15_model_features.png"),
        "Top 15 SHAP importance (model features)",
        topn=15
    )

    # 2) 聚合回原始特征
    mapper = build_original_feature_map(original_feature_cols)
    shap_df_orig = aggregate_shap_to_original_features(
        shap_df_model=shap_df_model,
        original_feature_cols=original_feature_cols,
        mapper=mapper
    )

    orig_mean_abs = shap_df_orig.abs().mean().sort_values(ascending=False).reset_index()
    orig_mean_abs.columns = ["feature", "mean_abs_shap"]
    orig_mean_abs.to_csv(
        os.path.join(out_dir, "shap_mean_abs_importance_original_features.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    test_index = X_test2.index
    X_orig_test = model_df.loc[test_index, original_feature_cols].copy()

    shap.summary_plot(
        shap_df_orig.values,
        X_orig_test[original_feature_cols],
        show=False
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(out_dir, "shap_summary_original_features.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    shap.summary_plot(
        shap_df_orig.values,
        X_orig_test[original_feature_cols],
        plot_type="bar",
        show=False
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(out_dir, "shap_bar_original_features.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    plot_top_mean_abs_shap(
        orig_mean_abs,
        os.path.join(out_dir, "shap_top15_original_features.png"),
        "Top 15 SHAP importance (original features)",
        topn=15
    )

    # 3) 原始特征 dependence图
    top_orig_features = orig_mean_abs["feature"].head(6).tolist()
    for feat in top_orig_features:
        plt.figure(figsize=(6, 4.5))
        plt.scatter(
            X_orig_test[feat].values,
            shap_df_orig[feat].values,
            alpha=0.7
        )
        plt.xlabel(feat)
        plt.ylabel(f"SHAP value of {feat}")
        plt.title(f"Dependence plot: {feat}")
        plt.tight_layout()
        plt.savefig(
            os.path.join(out_dir, f"dependence_{feat}.png"),
            dpi=300,
            bbox_inches="tight"
        )
        plt.close()


# =========================
# Error analysis
# =========================

In [ ]:
def run_error_stratification(y_true, y_pred, out_dir):
    df_err = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred
    })
    df_err["abs_error"] = np.abs(df_err["y_pred"] - df_err["y_true"])
    df_err["ape"] = np.where(df_err["y_true"] > 0, df_err["abs_error"] / df_err["y_true"], np.nan)
    df_err["y_true_bin"] = pd.qcut(df_err["y_true"], q=4, duplicates="drop")

    err_summary = df_err.groupby("y_true_bin", observed=False).agg({
        "y_true": ["count", "mean"],
        "abs_error": ["mean", "median"],
        "ape": ["mean", "median"]
    })
    err_summary.columns = ["_".join(col).strip() for col in err_summary.columns]
    err_summary = err_summary.reset_index()

    err_summary.to_csv(
        os.path.join(out_dir, "error_stratification_by_true_quantile.csv"),
        index=False,
        encoding="utf-8-sig"
    )


# =========================
# Plot helpers
# =========================

In [ ]:
def plot_pred_scatter(y_true, y_pred, save_path, title):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, alpha=0.75)
    mn = min(y_true.min(), y_pred.min())
    mx = max(y_true.max(), y_pred.max())
    plt.plot([mn, mx], [mn, mx], linestyle="--")
    plt.xlabel("Observed")
    plt.ylabel("Predicted")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()


In [ ]:
def plot_residual(y_true, y_pred, save_path, title):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    residual = y_pred - y_true
    plt.figure(figsize=(7, 5))
    plt.scatter(y_pred, residual, alpha=0.75)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Predicted")
    plt.ylabel("Residual (Pred - Obs)")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()


In [ ]:
def plot_cv_box(df_cv, save_path, title):
    plt.figure(figsize=(6, 5))
    plt.boxplot(df_cv["log_R2"].values, vert=True)
    plt.ylabel("log_R2")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()


In [ ]:
def plot_top_gain(gain_summary, save_path, topn=15):
    if len(gain_summary) == 0:
        return
    g = gain_summary.head(topn).iloc[::-1]
    plt.figure(figsize=(8, max(5, 0.35 * len(g))))
    plt.barh(g["feature"], g["gain"])
    plt.xlabel("Mean gain")
    plt.title("Top feature importance by gain")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()


# =========================
# Main
# =========================

In [ ]:
def main():
    ensure_outdir(OUT_DIR)

    df, fp = load_data()
    print("原始列名：", df.columns.tolist())

    X, y, meta, sample_col, target_col, model_df, feature_cols, original_feature_cols, clean_stats = clean_and_prepare_data(df)

    pd.DataFrame({"used_feature": feature_cols}).to_csv(
        os.path.join(OUT_DIR, "used_features.csv"),
        index=False,
        encoding="utf-8-sig"
    )
    pd.DataFrame({"original_feature": original_feature_cols}).to_csv(
        os.path.join(OUT_DIR, "original_features.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    with open(os.path.join(OUT_DIR, "cleaning_stats.json"), "w", encoding="utf-8") as f:
        json.dump(to_py(clean_stats), f, ensure_ascii=False, indent=2)

    print(f"文件: {fp}")
    print(f"sample列: {sample_col}")
    print(f"target列: {target_col}")
    print(f"原始行数: {clean_stats['raw_rows']}")
    print(f"去重后样本数: {clean_stats['after_drop_duplicates']}")
    print(f"特征数: {X.shape[1]}")

    final_cv, cv_feature_freq, cv_dropped_corr_freq = repeated_cv_eval(
        X=X,
        y=y,
        params=BEST_PARAMS,
        n_splits=FINAL_CV_SPLITS,
        n_repeats=FINAL_CV_REPEATS,
        num_boost_round=NUM_BOOST_ROUND_FINAL_CV
    )

    final_cv.to_csv(
        os.path.join(OUT_DIR, "final_random_cv_results.csv"),
        index=False,
        encoding="utf-8-sig"
    )
    cv_feature_freq.to_csv(
        os.path.join(OUT_DIR, "final_random_cv_selected_feature_frequency.csv"),
        index=False,
        encoding="utf-8-sig"
    )
    cv_dropped_corr_freq.to_csv(
        os.path.join(OUT_DIR, "final_random_cv_dropped_high_corr_frequency.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    hold = holdout_eval_random(X, y, meta, BEST_PARAMS)

    holdout_pred = hold["meta_test"].copy()
    holdout_pred["y_true"] = hold["y_test"]
    holdout_pred["y_pred"] = hold["y_test_pred"]
    holdout_pred.to_csv(
        os.path.join(OUT_DIR, "holdout_pred.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    hold["selected_features_df"].to_csv(
        os.path.join(OUT_DIR, "selected_features_bagging_frequency.csv"),
        index=False,
        encoding="utf-8-sig"
    )
    hold["gain_summary"].to_csv(
        os.path.join(OUT_DIR, "feature_importance_gain.csv"),
        index=False,
        encoding="utf-8-sig"
    )
    pd.DataFrame({
        "dropped_high_corr_feature": hold["dropped_corr"]
    }).to_csv(
        os.path.join(OUT_DIR, "holdout_dropped_high_corr_features.csv"),
        index=False,
        encoding="utf-8-sig"
    )
    pd.DataFrame({
        "bag_id": np.arange(1, len(hold["best_iterations"]) + 1),
        "best_iteration": hold["best_iterations"]
    }).to_csv(
        os.path.join(OUT_DIR, "holdout_best_iterations.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    plot_pred_scatter(
        hold["y_test"], hold["y_test_pred"],
        os.path.join(OUT_DIR, "holdout_pred_scatter.png"),
        "Random Holdout Observed vs Predicted"
    )
    plot_residual(
        hold["y_test"], hold["y_test_pred"],
        os.path.join(OUT_DIR, "holdout_residual_plot.png"),
        "Random Holdout Residual Plot"
    )
    plot_cv_box(
        final_cv,
        os.path.join(OUT_DIR, "cv_logR2_boxplot.png"),
        "Repeated Random CV log_R2"
    )
    plot_top_gain(
        hold["gain_summary"],
        os.path.join(OUT_DIR, "top_gain_importance.png"),
        topn=15
    )

    run_shap_analysis_original(
        saved_artifacts=hold["saved_artifacts"],
        out_dir=OUT_DIR,
        model_df=model_df,
        sample_col=sample_col,
        target_col=target_col,
        original_feature_cols=original_feature_cols
    )
    run_error_stratification(hold["y_test"], hold["y_test_pred"], OUT_DIR)

    summary_obj = {
        "file": fp,
        "sample_col": sample_col,
        "target_col": target_col,
        "n_rows_raw": int(clean_stats["raw_rows"]),
        "n_samples_after_cleaning": int(len(y)),
        "n_features_model": int(X.shape[1]),
        "n_features_original": int(len(original_feature_cols)),
        "best_params": BEST_PARAMS,
        "final_random_cv_mean_log_R2": float(final_cv["log_R2"].mean()),
        "final_random_cv_std_log_R2": float(final_cv["log_R2"].std()),
        "final_random_cv_p10_log_R2": float(np.percentile(final_cv["log_R2"], 10)),
        "final_random_cv_mean_R2": float(final_cv["R2"].mean()),
        "final_random_cv_mean_MAE": float(final_cv["MAE"].mean()),
        "final_random_cv_mean_RMSE": float(final_cv["RMSE"].mean()),
        "final_random_holdout_train": hold["train_metrics"],
        "final_random_holdout_test": hold["test_metrics"]
    }

    with open(os.path.join(OUT_DIR, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(to_py(summary_obj), f, ensure_ascii=False, indent=2)

    print("\n===== Final Random CV =====")
    print("mean log_R2 = {:.4f} ± {:.4f}".format(final_cv["log_R2"].mean(), final_cv["log_R2"].std()))
    print("p10  log_R2 = {:.4f}".format(np.percentile(final_cv["log_R2"], 10)))
    print("mean R2     = {:.4f}".format(final_cv["R2"].mean()))

    print("\n===== Final Random Holdout =====")
    print("Train log_R2 = {:.4f}".format(hold["train_metrics"]["log_R2"]))
    print("Test  log_R2 = {:.4f}".format(hold["test_metrics"]["log_R2"]))
    print("Test  R2     = {:.4f}".format(hold["test_metrics"]["R2"]))

    print("\n结果已保存到:", OUT_DIR)



In [ ]:
# ===== 运行 SHAP（自带训练，无需先跑建模 notebook）=====
SHAP_DIR = os.path.join(OUT_DIR, "shap_particle")
os.makedirs(SHAP_DIR, exist_ok=True)

print("颗粒相: 复现训练以拿到 saved_artifacts ...")
df, fp = load_data()
X, y, meta, sample_col, target_col, model_df, feature_cols, original_feature_cols, clean_stats = clean_and_prepare_data(df)
hold = holdout_eval_random(X, y, meta, BEST_PARAMS)
saved_artifacts = hold["saved_artifacts"]

print("颗粒相: 运行 SHAP (模型特征层 + 原始特征层) ...")
run_shap_analysis_original(
    saved_artifacts=saved_artifacts,
    out_dir=SHAP_DIR,
    model_df=model_df,
    sample_col=sample_col,
    target_col=target_col,
    original_feature_cols=original_feature_cols,
)
print("颗粒相 SHAP 完成, 输出:", SHAP_DIR)
